### libraries

In [2]:
import requests
import importlib
import numpy as np
import fastf1
from pathlib import Path
import pandas as pd

### load dataframes

In [3]:
working_directory = Path.cwd().parent.parent
print(working_directory)

file_path_1 = working_directory / "data" / "clean" / "driver_points.csv"
file_path_2 = working_directory / "data" / "clean" / "driver_price.csv"
file_path_3 = working_directory / "data" / "clean" / "driver_roster.csv"
file_path_4 = working_directory / "data" / "clean" / "placement_table.csv"



c:\Users\jackg\code\F1Fantasy


In [4]:
driver_points = pd.read_csv(file_path_1).drop(columns=["Unnamed: 0"])
driver_price = pd.read_csv(file_path_2).drop(columns=["Unnamed: 0"])
driver_roster = pd.read_csv(file_path_3).drop(columns=["Unnamed: 0"])
driver_placement = pd.read_csv(file_path_4).drop(columns=["Unnamed: 0"])

In [5]:
driver_points.tail()

,driver,race,points,year,race_name,start_date,location,country_code,meeting_key,month,start_epoch
1671,BOR,6,NaN,2026,Miami Grand Prix,2026-05-03 20:00:00+00:00,Miami Gardens,USA,1284,5,1777838400
1672,LIN,6,NaN,2026,Miami Grand Prix,2026-05-03 20:00:00+00:00,Miami Gardens,USA,1284,5,1777838400
1673,COL,6,NaN,2026,Miami Grand Prix,2026-05-03 20:00:00+00:00,Miami Gardens,USA,1284,5,1777838400
1674,PER,6,NaN,2026,Miami Grand Prix,2026-05-03 20:00:00+00:00,Miami Gardens,USA,1284,5,1777838400
1675,BOT,6,NaN,2026,Miami Grand Prix,2026-05-03 20:00:00+00:00,Miami Gardens,USA,1284,5,1777838400


In [6]:
driver_price.head()

,driver,race,price,year,race_name,start_date,location,country_code,meeting_key,month,start_epoch
0,VER,1,26.9,2023,Bahrain Grand Prix,2023-03-05 15:00:00+00:00,Sakhir,BRN,1141,3,1678028400
1,HAM,1,23.7,2023,Bahrain Grand Prix,2023-03-05 15:00:00+00:00,Sakhir,BRN,1141,3,1678028400
2,LEC,1,21.2,2023,Bahrain Grand Prix,2023-03-05 15:00:00+00:00,Sakhir,BRN,1141,3,1678028400
3,PER,1,18.0,2023,Bahrain Grand Prix,2023-03-05 15:00:00+00:00,Sakhir,BRN,1141,3,1678028400
4,RUS,1,18.6,2023,Bahrain Grand Prix,2023-03-05 15:00:00+00:00,Sakhir,BRN,1141,3,1678028400


In [7]:
driver_roster.head()

,driver,race,constructor,year
0,VER,1,RBR,2023
1,PER,1,RBR,2023
2,NOR,1,MCL,2023
3,PIA,1,MCL,2023
4,HAM,1,MER,2023


In [8]:
driver_roster.columns

Index(['driver', 'race', 'constructor', 'year'], dtype='object')

In [9]:
driver_price.columns

Index(['driver', 'race', 'price', 'year', 'race_name', 'start_date',
       'location', 'country_code', 'meeting_key', 'month', 'start_epoch'],
      dtype='object')

In [10]:
driver_points.columns

Index(['driver', 'race', 'points', 'year', 'race_name', 'start_date',
       'location', 'country_code', 'meeting_key', 'month', 'start_epoch'],
      dtype='object')

In [11]:
driver_placement.columns

Index(['date_start', 'driver', 'driver_number', 'position', 'points',
       'gap_to_leader', 'meeting_key', 'session_key', 'dnf', 'dns', 'dsq',
       'nc'],
      dtype='object')

In [12]:
#duplicate rows for the merge
driver_price = driver_price.drop(columns = ["race_name", "start_date", "location", "country_code", "month", "start_epoch"])

In [13]:
# two left joins to create the master df
driver_price_points = driver_points.merge(driver_price, how="left", on=["driver", "race", "year"])
driver_price_points = driver_price_points.merge(driver_roster, how="left", on=["year", "driver", "race"])

#renaming
driver_price_points['race_num'] = driver_price_points['race']
driver_price_points = driver_price_points.drop(columns = {"race"})

#some reordering for organizaiton sake
cols = ["year", "race_num", "race_name", "location", "start_date", "driver", "price", "points", "constructor"]
driver_price_points = driver_price_points[cols + [c for c in driver_price_points.columns if c not in cols]]

driver_price_points.head()

,year,race_num,race_name,location,start_date,driver,price,points,constructor,country_code,meeting_key_x,month,start_epoch,meeting_key_y
0,2023,1,Bahrain Grand Prix,Sakhir,2023-03-05 15:00:00+00:00,VER,26.9,35.0,RBR,BRN,1141,3,1678028400,1141
1,2023,1,Bahrain Grand Prix,Sakhir,2023-03-05 15:00:00+00:00,PER,18.0,28.0,RBR,BRN,1141,3,1678028400,1141
2,2023,1,Bahrain Grand Prix,Sakhir,2023-03-05 15:00:00+00:00,HAM,23.7,19.0,MER,BRN,1141,3,1678028400,1141
3,2023,1,Bahrain Grand Prix,Sakhir,2023-03-05 15:00:00+00:00,NOR,11.2,-1.0,MCL,BRN,1141,3,1678028400,1141
4,2023,1,Bahrain Grand Prix,Sakhir,2023-03-05 15:00:00+00:00,ALO,8.3,39.0,AMR,BRN,1141,3,1678028400,1141


### Price Based Features
- price_change_previous_race - done
- price_rank - done
- points_per_mill_last_three

In [14]:
driver_price_points = driver_price_points.sort_values(["driver","start_date"])
driver_price_points["price_change_prev_race"] = driver_price_points.groupby("driver")["price"].diff().round(2)
driver_price_points["price_rank"] = driver_price_points.groupby(["year","race_num"])["price"].rank(ascending=False, method="dense")


In [15]:
driver_price_points["year"].value_counts()

year
2024    648
2025    552
2023    484
2026     88
Name: count, dtype: int64

### Points Based Features
- points_last_three_avg - done
- points_last_five_avg - done
-ppm average for the last three races
- ppm average for the last 5 races
-season total ppm



In [16]:
driver_price_points = driver_price_points.sort_values(["driver","year","race_num"]).copy()

driver_price_points["points_last_three_avg"] = (
    driver_price_points.groupby("driver")["points"]
      .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
      .round(2)
)

driver_price_points["points_last_five_avg"] = (
    driver_price_points.groupby("driver")["points"]
      .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
      .round(2)
)


driver_price_points["ppm_last_3"] = (driver_price_points["points_last_three_avg"] / driver_price_points["price"].replace(0, np.nan)).round(2)
driver_price_points["ppm_last_5"] = (driver_price_points["points_last_five_avg"] / driver_price_points["price"].replace(0, np.nan)).round(2)

In [17]:
driver_price_points['year'].value_counts()

year
2024    648
2025    552
2023    484
2026     88
Name: count, dtype: int64

In [18]:
driver_price_points = driver_price_points.sort_values(["year", "race_num", "price"], ascending=[True, True, False]) #back to original sort order
driver_price_points.tail(5)

,year,race_num,race_name,location,start_date,driver,price,points,constructor,country_code,meeting_key_x,month,start_epoch,meeting_key_y,price_change_prev_race,price_rank,points_last_three_avg,points_last_five_avg,ppm_last_3,ppm_last_5
1770,2026,6,Miami Grand Prix,Miami Gardens,2026-05-03 20:00:00+00:00,PER,7.0,NaN,CAD,USA,1284,5,1777838400,1284,0.6,17.0,9.33,-1.0,1.33,-0.14
1762,2026,6,Miami Grand Prix,Miami Gardens,2026-05-03 20:00:00+00:00,STR,6.2,NaN,AMR,USA,1284,5,1777838400,1284,-0.6,18.0,-18.00,-6.4,-2.90,-1.03
1767,2026,6,Miami Grand Prix,Miami Gardens,2026-05-03 20:00:00+00:00,BOR,5.8,NaN,AUD,USA,1284,5,1777838400,1284,-0.6,19.0,0.67,4.4,0.12,0.76
1765,2026,6,Miami Grand Prix,Miami Gardens,2026-05-03 20:00:00+00:00,HUL,5.0,NaN,AUD,USA,1284,5,1777838400,1284,-0.6,20.0,-1.00,0.6,-0.20,0.12
1771,2026,6,Miami Grand Prix,Miami Gardens,2026-05-03 20:00:00+00:00,BOT,4.1,NaN,CAD,USA,1284,5,1777838400,1284,-0.6,21.0,-3.67,-3.0,-0.90,-0.73


# Split to Hist and Current Week

### Dataset 1 will be used for the training of the model and will be further split into test and training sets

#### Filtering
- making sure that the only rows that exist are those that have points and cost for every record. The ones that do not have a price/points are not active for the given week anyway

In [19]:
hist_driver_price_points = driver_price_points[(~driver_price_points['price'].isna()) & (~driver_price_points['points'].isna())]

#### Exporting dataset 1

In [20]:
file_path = working_directory / "data" / "semi-clean" / "hist_driver_points_df_v1.csv"

hist_driver_price_points.to_csv(file_path)

### Dataset 2 will be used to create predictions for the current weeks race
- this should be held seperate as it has missing values

#### Exporting dataset 2

In [21]:

def get_current_week_race_num():
    today_dt = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
    #calendar load
    file_path = working_directory / "data" / "clean" / "race_session_meeting_info.csv"
    calendar = pd.read_csv(file_path).drop(columns=["Unnamed: 0"])

    #getting the current race number
    calendar["start_date"] = pd.to_datetime(calendar["start_date"], utc=True)
    next_race_int = calendar.loc[calendar["start_date"] >= today_dt, "race"].iloc[0].astype(int)

    #getting current year

    today_dt = pd.to_datetime(today_dt)
    year = today_dt.year


    return next_race_int, year



In [22]:
current_week_race_num, current_year = get_current_week_race_num()
print(f'Race: {current_week_race_num}, Year: {current_year}')

Race: 6, Year: 2026


In [23]:
current_week_driver_price_points = driver_price_points[(driver_price_points["race_num"] == current_week_race_num) & (driver_price_points["year"] == current_year)]

In [24]:
current_week_driver_price_points.tail()

,year,race_num,race_name,location,start_date,driver,price,points,constructor,country_code,meeting_key_x,month,start_epoch,meeting_key_y,price_change_prev_race,price_rank,points_last_three_avg,points_last_five_avg,ppm_last_3,ppm_last_5
1770,2026,6,Miami Grand Prix,Miami Gardens,2026-05-03 20:00:00+00:00,PER,7.0,NaN,CAD,USA,1284,5,1777838400,1284,0.6,17.0,9.33,-1.0,1.33,-0.14
1762,2026,6,Miami Grand Prix,Miami Gardens,2026-05-03 20:00:00+00:00,STR,6.2,NaN,AMR,USA,1284,5,1777838400,1284,-0.6,18.0,-18.00,-6.4,-2.90,-1.03
1767,2026,6,Miami Grand Prix,Miami Gardens,2026-05-03 20:00:00+00:00,BOR,5.8,NaN,AUD,USA,1284,5,1777838400,1284,-0.6,19.0,0.67,4.4,0.12,0.76
1765,2026,6,Miami Grand Prix,Miami Gardens,2026-05-03 20:00:00+00:00,HUL,5.0,NaN,AUD,USA,1284,5,1777838400,1284,-0.6,20.0,-1.00,0.6,-0.20,0.12
1771,2026,6,Miami Grand Prix,Miami Gardens,2026-05-03 20:00:00+00:00,BOT,4.1,NaN,CAD,USA,1284,5,1777838400,1284,-0.6,21.0,-3.67,-3.0,-0.90,-0.73


#### Exporting Dataset 2

In [25]:
file_path = working_directory / "data" / "semi-clean" / "current_week_driver_points_df_v1.csv"

current_week_driver_price_points.to_csv(file_path)